# LoveDA training on Colab

This notebook runs the whole pipeline on a free GPU. It gets the code, installs the extra dependencies, downloads LoveDA, trains DeepLabV3 ResNet50, then evaluates, analyzes and shows the results.

Before running, switch the runtime to a GPU. Use the menu Runtime then Change runtime type then pick a GPU such as T4.

Colab sessions are temporary, so the last cell copies your run to Google Drive. Do that or you lose the checkpoint when the session ends.

## 1. Check the GPU

In [10]:
import torch
print("torch", torch.__version__, "cuda available", torch.cuda.is_available())
!nvidia-smi -L

torch 2.11.0+cu128 cuda available True
GPU 0: Tesla T4 (UUID: GPU-caae6f09-8311-e024-50fb-da5a29d2451e)


## 2. Get the code

Two ways. Clone from GitHub if you pushed the project, or upload a zip of the project folder. Run only one of the two cells below.

In [11]:
# Option A. Clone from GitHub.
REPO_URL = "https://github.com/ShanmukhUpad/semantic-segmentation-feature.git"
!git clone $REPO_URL
%cd semantic-segmentation-feature

Cloning into 'semantic-segmentation-feature'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 51 (delta 4), reused 51 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 46.41 KiB | 2.58 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/semantic-segmentation-feature/semantic-segmentation-feature


In [12]:
# Option B. Upload a zip of the project instead of cloning, then unzip it.
# from google.colab import files
# uploaded = files.upload()            # choose your project zip
# import zipfile, io
# name = next(iter(uploaded))
# with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
#     archive.extractall(".")
# %cd semantic-segmentation-feature

## 3. Install the extra dependencies

Colab already provides a CUDA build of torch and torchvision, so do not reinstall those. Installing the pinned CPU torch from requirements would replace the GPU build and slow everything down, so only the lighter extras are installed here.

In [13]:
!pip -q install seaborn tensorboard pyyaml tqdm scikit-learn gradio

## 4. Download LoveDA

This pulls Train and Val from the official Zenodo record and arranges them. It is a few gigabytes, so it takes a few minutes.

In [14]:
!python scripts/download_loveda.py --root data/raw/LoveDA

Traceback (most recent call last):
  File "/content/semantic-segmentation-feature/semantic-segmentation-feature/scripts/download_loveda.py", line 98, in <module>
    main()
  File "/content/semantic-segmentation-feature/semantic-segmentation-feature/scripts/download_loveda.py", line 71, in main
    download(ZIPS[split], zip_path)
  File "/content/semantic-segmentation-feature/semantic-segmentation-feature/scripts/download_loveda.py", line 32, in download
    with urllib.request.urlopen(url) as response:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py", line 215, in urlopen
    return opener.open(url, data, timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py", line 521, in open
    response = meth(req, response)
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py", line 630, in http_response
    response = self.parent.error(
               ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/pyt

## 5. Train on the GPU

This uses the LoveDA config. The overrides give a shorter first run that still produces a usable model. Raise train.epochs for a stronger result. On a T4 each epoch takes a few minutes.

In [6]:
!python scripts/train.py --config configs/default.yaml --set train.epochs=15 dataloader.batch_size=8 dataloader.num_workers=2

2026-06-22 14:31:20.372190: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/content/semantic-segmentation-feature/scripts/train.py", line 316, in <module>
    main()
  File "/content/semantic-segmentation-feature/scripts/train.py", line 213, in main
    train_loader, val_loader = build_dataloaders(cfg)
                               ^^^^^^^^^^^^^^^^^^^^^^
  File "/content/semantic-segmentation-feature/data/loader.py", line 20, in build_dataloaders
    train_dataset = build_dataset(cfg, "train")
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/semantic-segmentation-feature/data/registry.py", line 36, in build_dataset
    return DATASETS[name](cfg, split)
           ^^^^^^^^^^^^^^^^^^^^^^^

## 6. Evaluate and analyze on the validation split

LoveDA Test labels are withheld, so the validation split is the held out set.

In [7]:
!python scripts/evaluate.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --split val
!python scripts/analyze.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --split val

Traceback (most recent call last):
  File "/content/semantic-segmentation-feature/scripts/evaluate.py", line 168, in <module>
    main()
  File "/content/semantic-segmentation-feature/scripts/evaluate.py", line 105, in main
    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1530, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 795, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 776, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directo

## 7. View the results inline

In [8]:
import glob
from IPython.display import Image as IPImage, Markdown, display
base = "results/loveda_deeplabv3/analysis"
display(Markdown(open(base + "/report.md").read()))
for path in sorted(glob.glob(base + "/*.png")) + sorted(glob.glob(base + "/error_maps/*.png")):
    print(path)
    display(IPImage(path))

FileNotFoundError: [Errno 2] No such file or directory: 'results/loveda_deeplabv3/analysis/report.md'

## 8. Training curves and sample images in TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/loveda_deeplabv3/tensorboard

## 9. Interactive prediction app

Launch the upload GUI with a public link. Open the printed gradio.live URL, upload an aerial tile and see the real prediction. Stop the cell to shut the app down.

In [ ]:
!python scripts/app.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --share

## 10. Save the run to Google Drive

Colab sessions are temporary. Run this to keep the checkpoint, report and figures.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r results/loveda_deeplabv3 "/content/drive/MyDrive/loveda_deeplabv3"
print("copied run to Google Drive")